## Notebook setup
This bootstrap finds the repository root when Jupyter starts in the project directory or inside `notebooks/`, adds it to Python's import path, and uses it as the working directory so generated artifact paths are consistent.


In [1]:
from pathlib import Path
import os
import sys

search_from = Path.cwd().resolve()
project_root = next((candidate for candidate in (search_from, *search_from.parents)
                     if (candidate / 'analyzer').is_dir()
                     and (candidate / 'data' / 'job_dataset.csv').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Could not find the SkillBridge repository root. Start Jupyter from the project folder or its notebooks/ folder.')
sys.path.insert(0, str(project_root))
os.chdir(project_root)
print(f'Repository root: {project_root}')

Repository root: C:\Users\SARTHAK BASNET\OneDrive\Desktop\Ai-skill-gap-analyzer


# SkillBridge skill matcher: training from scratch

This submission trains a Gensim Word2Vec model only on `data/job_dataset.csv`. No pretrained tokenizer, embedding, model, or weights are loaded. The checked-in hand-labelled fixture is used as-is for evaluation. Run from the repository root after installing `requirements.txt`.


## 1. Corpus and cleaning
Each CSV row becomes one document from Title, Skills, Responsibilities, and Keywords. Skills and Keywords use semicolon-delimited lists; responsibilities remain text. Cleaning lowercases and extracts contiguous ASCII letters/digits, treating punctuation as token boundaries. This deliberately simple tokenizer strips punctuation and does not stem, lemmatize, or remove stop words.


In [2]:
from analyzer.train_matcher import load_corpus
from collections import Counter
documents, total_word_count = load_corpus()
cleaning_vocabulary = Counter(token for document in documents for token in document)
print(f'Documents: {len(documents):,}')
print(f'Total words after cleaning: {total_word_count:,}')
print(f'Vocabulary before Word2Vec min_count filtering: {len(cleaning_vocabulary):,}')

Documents: 1,068
Total words after cleaning: 74,081
Vocabulary before Word2Vec min_count filtering: 1,740


## 2. Train Word2Vec from random initialization
Skip-gram (`sg=1`) is used to learn local word-context relationships from the relatively small corpus. A 100-dimensional space, window 5, negative sampling, and 10 epochs are modest starting values. `min_count=2` filters singleton noise. Four workers follow the requested starting setup; worker scheduling can cause small run-to-run numerical differences. The model's vocabulary size and elapsed training time are reported below.


In [3]:
from analyzer.train_matcher import train_and_export
training_report = train_and_export()
training_report

{'documents': 1068,
 'word_count': 74081,
 'vocabulary_after_cleaning': 1740,
 'training_vocabulary': 1487,
 'training_seconds': 1.027077200000349,
 'parameters': {'vector_size': 100,
  'window': 5,
  'min_count': 2,
  'workers': 4,
  'sg': 1,
  'seed': 17,
  'negative': 5,
  'sample': 0.001,
  'epochs': 10},
 'no_embedding_skills': ['Calculus', 'Message Queues'],
 'model_path': 'C:\\Users\\SARTHAK BASNET\\OneDrive\\Desktop\\Ai-skill-gap-analyzer\\models\\word2vec_scratch.model',
 'embeddings_path': 'C:\\Users\\SARTHAK BASNET\\OneDrive\\Desktop\\Ai-skill-gap-analyzer\\models\\skill_embeddings.json'}

## 3. Phrase and canonical skill vectors
A phrase vector is the arithmetic mean of its known token vectors; unknown words are ignored. If no token is known, the phrase has no vector and is rejected. A canonical skill with no in-vocabulary words is stored explicitly as `no embedding available` (`null` in the JSON artifact) and excluded from cosine ranking. Averaging loses word order and compositional meaning and is not equivalent to a contextual sentence embedding.


In [4]:
import json
from pathlib import Path
status = json.loads(Path(training_report['embeddings_path']).read_text(encoding='utf-8'))['skill_embedding_status']
unavailable = [name for name, value in status.items() if value['status'] == 'no embedding available']
print(f'Canonical skills: {len(status)}; available: {len(status)-len(unavailable)}; unavailable: {len(unavailable)}')
unavailable

Canonical skills: 67; available: 65; unavailable: 2


['Calculus', 'Message Queues']

## 4. Evaluate against the existing labelled fixture
Top-1 accuracy counts a prediction as correct only when its highest-scoring skill equals the label and it clears the threshold. Thresholds are swept on this same labelled fixture for descriptive calibration, so the selected threshold is not an independent held-out estimate. Report the full-fixture score alongside the historical pretrained-model result of 89.7% (52/58); the sample sizes differ. Inspect incorrect predictions and missing-vector cases rather than treating cosine scores as calibrated probabilities.


In [5]:
from analyzer.train_matcher import evaluate_thresholds
threshold_scores, evaluation_records = evaluate_thresholds()
best_threshold = max(threshold_scores, key=threshold_scores.get)
print(f'At the app default threshold 0.50: {threshold_scores[0.50]:.1%} ({round(threshold_scores[0.50]*len(evaluation_records))}/{len(evaluation_records)})')
print(f'Best threshold on this same fixture: {best_threshold:.2f}; top-1 accuracy: {threshold_scores[best_threshold]:.1%} ({round(threshold_scores[best_threshold]*len(evaluation_records))}/{len(evaluation_records)})')
print(f'Historical pretrained baseline: 89.7% (52/58), on a smaller earlier subset.')
[(r['text'], r['expected'], r['predicted'], round(r['confidence'], 3)) for r in evaluation_records if r['expected'] != r['predicted']][:30]

At the app default threshold 0.50: 50.6% (78/154)
Best threshold on this same fixture: 0.30; top-1 accuracy: 50.6% (78/154)
Historical pretrained baseline: 89.7% (52/58), on a smaller earlier subset.


[('worked with Amazon Web Services cloud infrastructure',
  'AWS',
  'Cloud Architecture',
  0.74),
 ('automated workflows with the Python programming language',
  'Python',
  'Data Structures and Algorithms',
  0.657),
 ('built interactive web pages with JavaScript',
  'JavaScript',
  'TypeScript',
  0.652),
 ('designed a scalable distributed system architecture',
  'System Design',
  'Cloud Architecture',
  0.843),
 ('designed a star schema for analytics',
  'Data Warehousing',
  'Data Structures and Algorithms',
  0.642),
 ('managed data quality and ownership policies',
  'Data Governance',
  'Data Structures and Algorithms',
  0.742),
 ('performed statistical analysis', 'Statistics', 'Business Analysis', 0.894),
 ('modeled uncertainty and probability with random variables',
  'Probability',
  'Data Structures and Algorithms',
  0.799),
 ('used matrices and vector operations',
  'Linear Algebra',
  'Monitoring and Observability',
  0.739),
 ('trained machine learning predictive mode

In [6]:
# Save a durable per-example evaluation record for error analysis/reproducibility.
Path('models/evaluation_records.json').write_text(json.dumps({'selected_threshold': best_threshold, 'threshold_scores': threshold_scores, 'records': evaluation_records}, indent=2), encoding='utf-8')

30225

## 5. Artifacts and interpretation
The notebook writes `models/word2vec_scratch.model`, `models/skill_embeddings.json`, and `models/evaluation_records.json`. These are generated locally and ignored by Git. The Django API's existing startup singleton loads the model and skill vectors once per process. Results should be interpreted as transfer-free learning from a limited job corpus: rare taxonomy terms, acronyms, short jargon, and professional skills absent from postings are expected failure modes. Re-run all cells after changing corpus or training parameters, then commit the notebook with its executed outputs for the project submission.
